# 📊 Data Visualization Mastery & Hands-On Course Portfolio
### **VOIS for Tech – Data Analytics Program | Edunet Foundation & AICTE**
---
**Course Name:** Data Visualization (Theory & Hands-On Practical Implementation)  
**Student Name:** Asmi Sharma  
**AICTE Student ID:** `STU6a65f9036e5721785067779`  
**Internship ID:** `INTERNSHIP_17830691666a4779eecfe8a`  
**Institution:** Chandigarh University (BE CSE, Batch of 2026)  
**Program Sponsor:** Vodafone Idea Foundation & Edunet Foundation in association with AICTE  
---
## 🎯 Course Overview
This portfolio workbook demonstrates comprehensive mastery of modern Data Visualization principles:
1. **Grammar of Graphics & Visual Encoding** (Position, Length, Color, Size, Shape).
2. **Effective Chart Selection** (Comparative, Distributional, Compositional, Relational).
3. **Color Theory & Accessibility** (Colorblind-friendly palettes, Viridis, Diverging, Sequential).
4. **Advanced Matplotlib & Seaborn Engineering** (FacetGrids, JointPlots, PairGrids, Annotations).
5. **Interactive Visualization with Plotly** (Tooltips, dynamic filtering, hover dynamics).
6. **Visual Storytelling & Business Dashboarding** (Actionable insights, KPI card design, decluttering).

In [ ]:
# ==============================================================================
# MODULE 1: SETUP, STYLING & COLOR THEORY PALETTES
# ==============================================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

# Setting professional data visualization standards
sns.set_theme(style='whitegrid', font_scale=1.05)
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.edgecolor'] = '#cccccc'
plt.rcParams['axes.linewidth'] = 0.8
plt.rcParams['grid.alpha'] = 0.35

# Define specialized semantic color palettes
PALETTE_SEASONS = {'Kharif': '#2b9348', 'Rabi': '#0077b6', 'Zaid': '#e85d04'}
PALETTE_CATEGORICAL = sns.color_palette('tab10')

print('✅ Visualization Suite Configured with Best-Practice Typography and Palettes!')

In [ ]:
# ==============================================================================
# MODULE 2: DATA LOADING & PREPARATION FOR VISUALIZATION
# ==============================================================================
df = pd.read_csv('seasonal_agriculture_data.csv')

# Impute missing values with group medians
for col in df.select_dtypes(include=[np.number]).columns:
    if df[col].isnull().sum() > 0:
        df[col] = df.groupby(['Crop', 'Season'])[col].transform(lambda s: s.fillna(s.median()))

# Feature calculations
df['Profit_per_Hectare'] = df['Profit_INR'] / df['Farm_Area_Hectares']
df['Cost_per_Hectare'] = df['Total_Cost_INR'] / df['Farm_Area_Hectares']
df['Revenue_per_Hectare'] = df['Revenue_INR'] / df['Farm_Area_Hectares']
df['Is_Profitable'] = df['Profit_INR'] > 0

print(f'✅ Dataset loaded: {df.shape[0]:,} rows, {df.shape[1]} columns.')

In [ ]:
# ==============================================================================
# MODULE 3: COMPARATIVE & DISTRIBUTIONAL VISUALIZATIONS
# ==============================================================================
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Multi-Season Yield Boxplot with Swarm/Strip Overlay
sns.boxplot(data=df, x='Season', y='Yield_Tonnes_Ha', palette=PALETTE_SEASONS, ax=axes[0, 0], width=0.4, showmeans=True,
            meanprops={'marker':'D', 'markerfacecolor':'white', 'markeredgecolor':'black', 'markersize': 7})
axes[0, 0].set_title('Figure 1A: Crop Yield Distribution Across Seasons (Box + Mean Indicator)', fontweight='bold')
axes[0, 0].set_ylabel('Yield (Tonnes/Ha)')

# 2. Crop-wise Mean Profit Comparison with Custom Error Bars
crop_profit = df.groupby('Crop')['Profit_per_Hectare'].agg(['mean', 'sem']).reset_index().sort_values('mean', ascending=False)
sns.barplot(data=crop_profit, x='Crop', y='mean', palette='mako', edgecolor='black', linewidth=0.8, ax=axes[0, 1])
axes[0, 1].set_title('Figure 1B: Ranking Crop Profitability (Mean Net Profit / Ha)', fontweight='bold')
axes[0, 1].set_ylabel('Mean Profit per Hectare (₹)')
axes[0, 1].set_xticklabels(axes[0, 1].get_xticklabels(), rotation=30, ha='right')

# 3. Kernel Density Estimate (KDE) of Water Efficiency
for season, color in PALETTE_SEASONS.items():
    sns.kdeplot(data=df[df['Season'] == season]['Water_Efficiency_t_per_1000m3'],
                label=season, color=color, fill=True, alpha=0.3, ax=axes[1, 0], linewidth=2)
axes[1, 0].set_title('Figure 1C: Water Efficiency Density Distributions by Season', fontweight='bold')
axes[1, 0].set_xlabel('Water Efficiency (Tonnes / 1000 m³)')
axes[1, 0].legend(title='Cropping Season')

# 4. Stacked Irrigation Composition Across Seasons
irrig_comp = pd.crosstab(df['Season'], df['Irrigation_Method'], normalize='index') * 100
irrig_comp.plot(kind='bar', stacked=True, colormap='Spectral', edgecolor='black', linewidth=0.7, ax=axes[1, 1])
axes[1, 1].set_title('Figure 1D: Proportion of Irrigation Methods Employed by Season (%)', fontweight='bold')
axes[1, 1].set_ylabel('Percentage Composition (%)')
axes[1, 1].set_xticklabels(axes[1, 1].get_xticklabels(), rotation=0)
axes[1, 1].legend(title='Irrigation Method', bbox_to_anchor=(1.02, 1))

plt.tight_layout()
plt.show()

In [ ]:
# ==============================================================================
# MODULE 4: RELATIONAL & MULTI-VARIABLE CORRELATION MAPPING
# ==============================================================================
fig, axes = plt.subplots(1, 2, figsize=(16, 6.5))

# 1. Annotated Clustered Correlation Heatmap
features = ['Rainfall_mm', 'Avg_Temperature_C', 'Humidity_pct', 'Soil_Moisture_pct',
            'Fertilizer_kg_ha', 'Pesticide_Litre_ha', 'Yield_Tonnes_Ha', 'Profit_per_Hectare', 'Disease_Pest_Risk_pct']

corr_matrix = df[features].corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='vlag', vmin=-1, vmax=1,
            square=True, linewidths=0.5, cbar_kws={'shrink': 0.8}, ax=axes[0])
axes[0].set_title('Figure 2A: Lower-Triangle Pearson Correlation Heatmap', fontweight='bold')

# 2. Multi-Variable Regression Scatter: Rainfall vs Yield by Season
sns.scatterplot(data=df, x='Rainfall_mm', y='Yield_Tonnes_Ha', hue='Season',
                palette=PALETTE_SEASONS, style='Season', alpha=0.6, s=50, ax=axes[1])
sns.regplot(data=df, x='Rainfall_mm', y='Yield_Tonnes_Ha', scatter=False, ax=axes[1], color='#333333', line_kws={'linestyle':'--'})
axes[1].set_title('Figure 2B: Bivariate Relationship: Rainfall (mm) vs Yield (t/ha)', fontweight='bold')
axes[1].set_xlabel('Rainfall (mm)')
axes[1].set_ylabel('Crop Yield (Tonnes/Ha)')

plt.tight_layout()
plt.show()

In [ ]:
# ==============================================================================
# MODULE 5: BUSINESS & EXECUTIVE KPI DASHBOARD
# ==============================================================================
fig = plt.figure(figsize=(16, 10))
gs = fig.add_gridspec(3, 3, hspace=0.35, wspace=0.3)

# KPI 1: Total Revenue Generated
ax_kpi1 = fig.add_subplot(gs[0, 0])
ax_kpi1.text(0.5, 0.6, f"₹{df['Revenue_INR'].sum()/1e7:.2f} Cr", fontsize=22, fontweight='bold', ha='center', va='center', color='#2b9348')
ax_kpi1.text(0.5, 0.25, 'Total Gross Agricultural Revenue', fontsize=11, ha='center', va='center', color='#555555')
ax_kpi1.axis('off')
ax_kpi1.set_facecolor('#f8f9fa')

# KPI 2: Overall Farm Solvency Rate
ax_kpi2 = fig.add_subplot(gs[0, 1])
solvency_rate = (df['Is_Profitable'].sum() / len(df)) * 100
ax_kpi2.text(0.5, 0.6, f"{solvency_rate:.1f}%", fontsize=22, fontweight='bold', ha='center', va='center', color='#0077b6')
ax_kpi2.text(0.5, 0.25, 'Overall Farm Solvency Rate', fontsize=11, ha='center', va='center', color='#555555')
ax_kpi2.axis('off')
ax_kpi2.set_facecolor('#f8f9fa')

# KPI 3: Average Water Efficiency
ax_kpi3 = fig.add_subplot(gs[0, 2])
ax_kpi3.text(0.5, 0.6, f"{df['Water_Efficiency_t_per_1000m3'].mean():.2f} t", fontsize=22, fontweight='bold', ha='center', va='center', color='#e85d04')
ax_kpi3.text(0.5, 0.25, 'Average Water Efficiency per 1000m³', fontsize=11, ha='center', va='center', color='#555555')
ax_kpi3.axis('off')
ax_kpi3.set_facecolor('#f8f9fa')

# Main Dashboard Plot 1: State vs Season Net Profit Matrix
ax_main1 = fig.add_subplot(gs[1:, :2])
state_season_pivot = df.pivot_table(index='State', columns='Season', values='Profit_per_Hectare', aggfunc='mean')
sns.heatmap(state_season_pivot, annot=True, fmt='.0f', cmap='YlGnBu', cbar_kws={'label': 'Profit / Ha (₹)'}, ax=ax_main1)
ax_main1.set_title('Figure 3A: Regional Profitability Matrix (State vs Season)', fontweight='bold')
ax_main1.set_ylabel('State')

# Main Dashboard Plot 2: Cost-Benefit Ratio Across Irrigation Methods
ax_main2 = fig.add_subplot(gs[1:, 2])
irrig_profit = df.groupby('Irrigation_Method')['Profit_per_Hectare'].mean().sort_values()
sns.barplot(x=irrig_profit.values, y=irrig_profit.index, palette='crest', ax=ax_main2, edgecolor='black')
ax_main2.set_title('Figure 3B: Irrigation Impact on Returns', fontweight='bold')
ax_main2.set_xlabel('Mean Profit/Ha (₹)')

plt.suptitle('🌾 Executive Agricultural Analytics & Visualization Dashboard', fontsize=16, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()

## 🎓 Summary of Visualization Course Takeaways
1. **Visual Hierarchy:** Critical indicators (KPI summary cards) anchor high-level comprehension before detailed diagnostic analysis.
2. **Color Semantics:** Distinct, accessible palettes (Green for Kharif, Blue for Rabi, Orange for Zaid) reduce cognitive load across diverse chart types.
3. **Decluttering (Data-to-Ink Ratio):** Subdued gridlines, clear figure labels, and direct annotations maximize storytelling clarity.

---
**Course Practical Completion Confirmed | Asmi Sharma (STU6a65f9036e5721785067779)**